In [2]:
import pandas as pd
import os 

os.getcwd()

os.listdir("../data/seoul_small_business")


['소상공인시장진흥공단_상가(상권)정보_서울_202403.csv',
 '소상공인시장진흥공단_상가(상권)정보_서울_202212.csv',
 '소상공인시장진흥공단_상가(상권)정보_서울_202312.csv',
 '소상공인시장진흥공단_상가(상권)정보_서울_202209.csv',
 '소상공인시장진흥공단_상가(상권)정보_서울_202409.csv',
 '소상공인시장진흥공단_상가(상권)정보_서울_202309.csv',
 '소상공인시장진흥공단_상가(상권)정보_서울_202306.csv',
 '소상공인시장진흥공단_상가(상권)정보_서울_202206.csv',
 '소상공인시장진흥공단_상가(상권)정보_서울_202109.csv',
 '소상공인시장진흥공단_상가(상권)정보_서울_202406.csv',
 '소상공인시장진흥공단_상가(상권)정보_서울_202112.csv',
 '소상공인시장진흥공단_상가(상권)정보_서울_202203.csv',
 '소상공인시장진흥공단_상가(상권)정보_서울_202303.csv']

In [4]:

import re
total = []
p = re.compile("[0-9]+")
for file in os.listdir("../data/seoul_small_business"):
    if( file[-3:] == "csv"):
        # print(file)
        try:
            tmp = pd.read_csv(f"../data/seoul_small_business/{file}", encoding='utf-8', low_memory=False)
        except Exception as e:
            tmp = pd.read_csv(f"../data/seoul_small_business/{file}", encoding='cp949', low_memory=False)
        tmp['날짜'] = p.findall(file)[0]
        total.append(tmp)

seoul_df = pd.concat(total, ignore_index=True)


In [6]:
seoul_df.columns

Index(['상가업소번호', '상호명', '지점명', '상권업종대분류코드', '상권업종대분류명', '상권업종중분류코드',
       '상권업종중분류명', '상권업종소분류코드', '상권업종소분류명', '표준산업분류코드', '표준산업분류명', '시도코드',
       '시도명', '시군구코드', '시군구명', '행정동코드', '행정동명', '법정동코드', '법정동명', '지번코드',
       '대지구분코드', '대지구분명', '지번본번지', '지번부번지', '지번주소', '도로명코드', '도로명', '건물본번지',
       '건물부번지', '건물관리번호', '건물명', '도로명주소', '구우편번호', '신우편번호', '동정보', '층정보',
       '호정보', '경도', '위도', '날짜'],
      dtype='object')

In [ ]:
seoul_df = seoul_df[seoul_df.상호명.notnull()].copy()

mask = seoul_df.상호명.apply(lambda x : True if x.find("스타벅스") > -1 else False)

seoul_df[mask].sort_values(by=['날짜']).head(3)


서울카페  = seoul_df[seoul_df.상권업종소분류명.isin(['카페','커피전문점/카페/다방'])]

import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'NanumGothic'


plt.style.use('ggplot')
fig, ax = plt.subplots(1,1, figsize=(15, 5))
ax.set_title("서울시 지역구별 카페 수", fontsize=20)
서울카페.날짜.value_counts().sort_index().plot( ax=ax)
plt.ylabel("매장수")
plt.xticks(rotation=45)
plt.show()

In [7]:
import requests 

url = "https://www.index.go.kr/unity/index/IndexTblGraphAjax.do"

payload = {'idxCd': '1058',
'sttsCd': '105801',
'chartOrd': '1',
'sDate': '202101',
'eDate': '202410'}

data = requests.post(url, data=payload).json()

csi = pd.DataFrame(data['modelAndView']['model']['resultList'])[['descDt', 'nmbrVal']]


In [8]:
csi

,descDt,nmbrVal
0,202101,95.3
1,202102,97.4
2,202103,100.6
3,202104,102.4
4,202105,105.5
5,202106,110.8
6,202107,103.6
7,202108,102.8
8,202109,104.1
9,202110,107.3


In [9]:
df = pd.read_csv(
    '../data/lotto1st.csv'
)
df

,회차,지점명,구분,주소
0,700,상호없음,자동,서울 광진구 구의동 589-10번지 대림아크로리버상가102
1,700,와이케이파트너스,자동,서울 동작구 사당동 265-29번지 1층
2,700,빛고을로또,수동,광주 광산구 소촌로 147 세븐일레븐
3,700,주택복권방,자동,"경기 안양시 만안구 수리산로 53,(안양동)"
4,700,평택포승점로또,자동,경기 평택시 여술로 24 CU편의점 내
...,...,...,...,...
3796,1068,복권명가,자동,강원 동해시 송정로 47-1 102호
3797,1068,로또스튜디오,자동,충북 청주시 청원구 사뜸로 48-1 1층
3798,1068,문화럭키 판매점,자동,충북 충주시 사직로 257-1 1층 2호
3799,1068,이마트24시 R신대부적점,수동,경북 경산시 대학로 386 1층


In [10]:
lotto_2 = pd.DataFrame(df.주소.value_counts()).reset_index()
lotto_2.columns =  ['주소', '당첨횟수']


In [21]:
lotto_2['rank'] = lotto_2['당첨횟수'].rank(method='min', ascending=False)
lotto_2

,주소,당첨횟수,rank
0,동행복권(dhlottery.co.kr),55,1.0
1,대구 달서구 대명천로 220 1층,20,2.0
2,충남 아산시 서해로 519-2,18,3.0
3,서울 노원구 동일로 1493 상계주공아파트(10단지) 주공10단지종합상가111,17,4.0
4,부산 기장군 정관중앙로 48 106호,14,5.0
...,...,...,...
2450,충남 아산시 음봉로 307 101호 CU편의점,1,735.0
2451,경기 안성시 진사길 31 우림아파트상가 B-03호,1,735.0
2452,경기 구리시 경춘로 223 CU구리명동점 내,1,735.0
2453,경기 고양시 일산서구 일청로 40-1 1층,1,735.0


In [23]:
hflight = pd.read_csv(
    '../data/hflight.csv'
)
hflight.columns

Index(['Year', 'Month', 'DayofMonth', 'DayOfWeek', 'DepTime', 'ArrTime',
       'UniqueCarrier', 'FlightNum', 'TailNum', 'ActualElapsedTime', 'AirTime',
       'ArrDelay', 'DepDelay', 'Origin', 'Dest', 'Distance', 'TaxiIn',
       'TaxiOut', 'Cancelled', 'CancellationCode', 'Diverted'],
      dtype='object')

In [26]:
pd.set_option('display.max_columns', None)

In [28]:
hflight.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 227496 entries, 0 to 227495
Data columns (total 21 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Year               227496 non-null  int64  
 1   Month              227496 non-null  int64  
 2   DayofMonth         227496 non-null  int64  
 3   DayOfWeek          227496 non-null  int64  
 4   DepTime            224591 non-null  float64
 5   ArrTime            224430 non-null  float64
 6   UniqueCarrier      227496 non-null  object 
 7   FlightNum          227496 non-null  int64  
 8   TailNum            226701 non-null  object 
 9   ActualElapsedTime  223874 non-null  float64
 10  AirTime            223874 non-null  float64
 11  ArrDelay           223874 non-null  float64
 12  DepDelay           224591 non-null  float64
 13  Origin             227496 non-null  object 
 14  Dest               227496 non-null  object 
 15  Distance           227496 non-null  int64  
 16  Ta

In [29]:
hflight

,Year,Month,DayofMonth,DayOfWeek,DepTime,ArrTime,UniqueCarrier,FlightNum,TailNum,ActualElapsedTime,AirTime,ArrDelay,DepDelay,Origin,Dest,Distance,TaxiIn,TaxiOut,Cancelled,CancellationCode,Diverted
0,2011,1,1,6,1400.0,1500.0,AA,428,N576AA,60.0,40.0,-10.0,0.0,IAH,DFW,224,7.0,13.0,0,NaN,0
1,2011,1,2,7,1401.0,1501.0,AA,428,N557AA,60.0,45.0,-9.0,1.0,IAH,DFW,224,6.0,9.0,0,NaN,0
2,2011,1,3,1,1352.0,1502.0,AA,428,N541AA,70.0,48.0,-8.0,-8.0,IAH,DFW,224,5.0,17.0,0,NaN,0
3,2011,1,4,2,1403.0,1513.0,AA,428,N403AA,70.0,39.0,3.0,3.0,IAH,DFW,224,9.0,22.0,0,NaN,0
4,2011,1,5,3,1405.0,1507.0,AA,428,N492AA,62.0,44.0,-3.0,5.0,IAH,DFW,224,9.0,9.0,0,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
227491,2011,12,6,2,1818.0,2111.0,WN,1191,N284WN,113.0,97.0,-9.0,8.0,HOU,TPA,781,5.0,11.0,0,NaN,0
227492,2011,12,6,2,2047.0,2334.0,WN,1674,N366SW,107.0,94.0,4.0,7.0,HOU,TPA,781,4.0,9.0,0,NaN,0
227493,2011,12,6,2,912.0,1031.0,WN,127,N777QC,79.0,61.0,-4.0,-3.0,HOU,TUL,453,4.0,14.0,0,NaN,0
227494,2011,12,6,2,656.0,812.0,WN,621,N727SW,76.0,64.0,-13.0,-4.0,HOU,TUL,453,3.0,9.0,0,NaN,0


1.문제
- 출발 공항에 대해서 도착 공항별로 평균 출발 지연시간 평균 도착지연시간을 구해서 DataFrame을 생성

2.문제 2
- 목적지 공항에 대해 연착 건수를 구하고, 연착 건수가 2000회 이상인 공항에 대한 데이터만 추출
- col -> Dest :목적지 공항 ArrDelay (연착은 5분이상)

3.문제 3
-  위의 결과를 바탕으로 목적지 공항 별 결항 횟수, 회항 횟수
- 운항 횟수를 구하시오 (Cancelled, Diverted, Air)
- 운항 횟수는 결항과 회항을 제외한 것 


In [78]:
# hflight[hflight['Origin'] == 'HOU']
# hflight['Origin'].unique() # ['IAH', 'HOU'],
hflight['Dest'].unique()

ans1 = hflight.groupby(['Origin', 'Dest'])[['ArrDelay', 'DepDelay']].mean()
ans1

ArrDelay   DepDelay
Origin Dest                      
HOU    ABQ    6.000987  11.581854
       ATL    6.810014   9.129112
       AUS    9.274145  12.188736
       BHM    6.672540  15.014599
       BKG  -16.233645  -3.201835
...                ...        ...
IAH    TUL    5.482379   5.738174
       TUS    7.801680   7.783871
       TYS   11.365915  10.170549
       VPS   12.457176  12.338691
       XNA    6.896277   6.690685

[149 rows x 2 columns]

In [66]:
tmp = hflight[hflight['ArrDelay'] < 5].groupby(['Origin', 'Dest'])[['ArrDelay']].agg(['count']).reset_index()
tmp.columns = ['Origin', 'Dest', 'Count']
ans2 = tmp[tmp['Count'] >= 2000]['Dest']
ans2.unique()

array(['DAL', 'MSY', 'ATL', 'AUS', 'CLT', 'CRP', 'DEN', 'DFW', 'EWR',
       'LAX', 'ORD', 'PHX'], dtype=object)

In [79]:
hflight['Cancelled'].unique() #array([0, 1])
hflight['Diverted'].unique()
tmp = hflight.groupby(['Origin', 'Dest'])[['Cancelled', 'Diverted']].agg(['sum',]).reset_index()
tmp.columns = ['Origin', 'Dest', 'Cancelled_count', 'Diverted_count']
tmp
tmp1 = hflight.groupby(['Origin', 'Dest'])[['FlightNum']].agg(['count']).reset_index()
tmp1.columns = ['Origin', 'Dest', 'Air']
tmp1
ans3 = pd.merge(tmp, tmp1, on=['Origin', 'Dest'], how='outer')
ans3['Air'] = ans3['Air'] - ans3['Cancelled_count'] - ans3['Diverted_count']
ans3.columns = ['Origin', "Dest", 'Cancelled', 'Diverted', 'Air']
ans3

,Origin,Dest,Cancelled,Diverted,Air
0,HOU,ABQ,5,1,1013
1,HOU,ATL,63,10,2816
2,HOU,AUS,5,2,1667
3,HOU,BHM,6,4,681
4,HOU,BKG,2,1,107
...,...,...,...,...,...
144,IAH,TUL,37,2,1816
145,IAH,TUS,15,2,1548
146,IAH,TYS,8,5,1197
147,IAH,VPS,10,6,864


In [ ]:
tmp = hflight.groupby(['Origin', 'Dest'])[['Cancelled', 'Diverted']].count()
tmp.reset_index()

0      1019
1      2889
2      1674
3       691
4       110
       ... 
144    1855
145    1565
146    1210
147     880
148    1172
Name: Diverted, Length: 149, dtype: int64